# 🎙️ Beatrice V2 Trainer — Custom Dataset (Kaggle Version)

Train your own voice conversion model using Beatrice V2 directly on Kaggle GPUs.

**Instructions:**
1. Run **Step 1** to install modules and configure directories in `/kaggle/working`.
2. Run **Option 1** or **Option 2** under **Step 2** to prepare your dataset.
3. Run **Step 3** to launch the Settings UI (requires Gradio public link or tunneling).
4. Run **Step 4** to launch TensorBoard.
5. Run **Step 5** to customize your training steps and run the training loop.
6. Run **Step 6** to generate and download output VST3 compatibility files or model checkpoints.
7. Run **Step 7** to clean up the working directory and free disk space.

> ⚠️ Make sure you have **GPU Accelerator** enabled in your Kaggle Notebook settings (right sidebar under 'Notebook options' -> 'Accelerator' -> T4 x2 or P100).

In [ ]:
# @title 1. Install Modules
import os
import shutil

# Safely reset directory
%cd /kaggle/working
if os.path.exists("/kaggle/working/beatrice-trainer"):
    shutil.rmtree("/kaggle/working/beatrice-trainer")

!pip install easy-colab-ui-extension > /dev/null 2>&1
!pip install colab-easy-ui2 pyworld==0.3.4 datasets soundfile librosa pandas tqdm > /dev/null 2>&1

# Clone from Hugging Face public repo which contains the full environment
!git clone https://huggingface.co/fierce-cats/beatrice-trainer /kaggle/working/beatrice-trainer

# Set up paths in Kaggle Working Directory for persistence
working_dir = "/kaggle/working/beatrice-trainer"
data_dir = "/kaggle/working/data"
output_dir = "/kaggle/working/output"
os.makedirs(data_dir, exist_ok=True)
os.makedirs(output_dir, exist_ok=True)

print("✅ Installation complete. Now choose Option 1 or Option 2 below to set up your dataset.")

## 📁 Step 2: Set Up Dataset
Choose either **Option 1** to pull a dataset from Hugging Face, or **Option 2** to import an uploaded Kaggle Input dataset.

In [ ]:
# @title 2a. Option 1: Download Dataset from Hugging Face
huggingface_repo_id = "meoconxinhxan/trump-speech-dataset-tts" # @param {type:"string"}
speaker_name = "trump" # @param {type:"string"}

import os
import glob
import shutil
import soundfile as sf
from datasets import load_dataset, Audio
from huggingface_hub import snapshot_download

# Define paths
data_dir = os.path.join("/kaggle/working/data", speaker_name)

# Ensure clean data directory
if os.path.exists(data_dir):
    shutil.rmtree(data_dir)
os.makedirs(data_dir, exist_ok=True)

print(f"📥 Downloading dataset from {huggingface_repo_id}...")

# Try loading via load_dataset first (good for parquet/hub audio datasets)
try:
    print("Attempting to load via Hugging Face datasets...")
    ds = load_dataset(huggingface_repo_id, split="train")
    
    # Identify audio column
    audio_col = None
    for col in ds.column_names:
        if isinstance(ds.features[col], Audio) or col in ['audio', 'path']:
            audio_col = col
            break
    
    if audio_col:
        ds = ds.cast_column(audio_col, Audio(sampling_rate=16000))
        print(f"Found audio column: {audio_col}. Extracting WAV files...")
        for i, item in enumerate(ds):
            audio_array = item[audio_col]['array']
            sr = item[audio_col]['sampling_rate']
            target_path = os.path.join(data_dir, f"{speaker_name}_{i}.wav")
            sf.write(target_path, audio_array, sr)
        print(f"✅ Extracted audio files to {data_dir}")
    else:
        raise ValueError("No audio column found in dataset features.")
        
except Exception as e_ds:
    print(f"Could not load as standard HF dataset: {e_ds}")
    print("Attempting direct repository download...")
    try:
        # Try snapshot_download to download wav files directly
        temp_download_dir = "/kaggle/working/temp_hf_download"
        if os.path.exists(temp_download_dir):
            shutil.rmtree(temp_download_dir)
        
        snapshot_download(
            repo_id=huggingface_repo_id,
            repo_type="dataset",
            local_dir=temp_download_dir,
            local_dir_use_symlinks=False
        )
        
        # Find and copy all WAV files
        wav_files = glob.glob(os.path.join(temp_download_dir, "**/*.wav"), recursive=True)
        if wav_files:
            for i, wf in enumerate(wav_files):
                shutil.copy2(wf, os.path.join(data_dir, f"{speaker_name}_{i}.wav"))
            print(f"✅ Downloaded and copied {len(wav_files)} WAV files to {data_dir}")
        else: 
            # Check if there are ZIP files
            zip_files = glob.glob(os.path.join(temp_download_dir, "**/*.zip"), recursive=True)
            if zip_files:
                import zipfile
                for zf_path in zip_files:
                    print(f"Extracting {os.path.basename(zf_path)}...")
                    with zipfile.ZipFile(zf_path, 'r') as zip_ref:
                        for member in zip_ref.namelist():
                            if member.endswith('/') or member.startswith('__MACOSX') or member.startswith('.'):
                                continue
                            if member.lower().endswith('.wav'):
                                basename = os.path.basename(member)
                                if basename:
                                    shutil.copyfileobj(zip_ref.open(member), open(os.path.join(data_dir, basename), 'wb'))
                print(f"✅ Extracted WAV files from ZIP to {data_dir}")
            else:
                print("❌ No WAV or ZIP files found in the repository.")
    except Exception as e_snap:
        print(f"❌ Failed to download from Hugging Face: {e_snap}")

total_files = len(glob.glob(os.path.join(data_dir, '*.wav')))
print(f"\nSetup complete. Total wav files in {speaker_name} folder: {total_files}")
if total_files > 0:
    print("🎉 Ready to train! Now run Step 3 (TensorBoard) or Step 4 (Training).")

In [ ]:
# @title 2b. Option 2: Copy Dataset from Kaggle Inputs
# @markdown **Instructions:** Upload your audio files (ZIP of WAVs or raw WAVs) as a Kaggle Dataset in the right panel (+ Add Input). Then, paste the directory path below.
kaggle_dataset_path = "/kaggle/input/your-dataset-name" # @param {type:"string"}
speaker_name = "my_voice" # @param {type:"string"}

import os
import glob
import shutil
import zipfile

# Define paths
data_dir = os.path.join("/kaggle/working/data", speaker_name)

# Ensure clean data directory
if os.path.exists(data_dir):
    shutil.rmtree(data_dir)
os.makedirs(data_dir, exist_ok=True)

if not os.path.exists(kaggle_dataset_path):
    print(f"❌ Path does not exist: {kaggle_dataset_path}. Please double check the Kaggle Input path.")
else:
    # Find ZIP files first
    zip_files = glob.glob(os.path.join(kaggle_dataset_path, "**/*.zip"), recursive=True)
    wav_files = glob.glob(os.path.join(kaggle_dataset_path, "**/*.wav"), recursive=True)

    if zip_files:
        for zf_path in zip_files:
            print(f"📦 Found ZIP file: {zf_path}. Extracting WAVs...")
            with zipfile.ZipFile(zf_path, 'r') as zf:
                for member in zf.namelist():
                    if member.endswith('/') or member.startswith('__MACOSX') or member.startswith('.'):
                        continue
                    if member.lower().endswith('.wav'):
                        basename = os.path.basename(member)
                        if basename:
                            target_path = os.path.join(data_dir, basename)
                            with zf.open(member) as src, open(target_path, 'wb') as dst:
                                dst.write(src.read())
    elif wav_files:
        print(f"🔊 Copying {len(wav_files)} WAV files directly...")
        for i, wf in enumerate(wav_files):
            basename = os.path.basename(wf)
            shutil.copy2(wf, os.path.join(data_dir, basename))
    else:
        print("❌ No WAV or ZIP files found in the specified Kaggle Input path.")

    # Count results
    total_files = len(glob.glob(os.path.join(data_dir, '*.wav')))
    print()
    print("=" * 50)
    print(f"✅ Import complete!")
    print(f"   Speaker:    {speaker_name}")
    print(f"   WAV files:  {total_files}")
    print("=" * 50)

    if total_files > 0:
        print("🚀 Ready to train! Run Step 3 (TensorBoard) or Step 4 (Training).")

In [ ]:
# @title 3. Start Settings UI (Optional)
# @markdown **Note:** Since Kaggle notebooks run behind a firewall, the local interface link will not open natively. You can skip this cell and run Step 4 directly, or configure a public Gradio tunnel.
import os
import torch
from colab_easy_ui.beatrice_trainer_main import wrapped_run_server
from IPython.display import HTML
from IPython.display import display
import portpicker
import time
import threading

data_dir = "/kaggle/working/data"
output_dir = "/kaggle/working/output"

def get_device():
  if torch.cuda.is_available():
    device = torch.cuda.get_device_name(0)
  else:
    device = "cpu"
  return device
device=get_device()

port = portpicker.pick_unused_port()
print(f"Starting Settings server on local port {port} in the background...")

def run_headless_server():
    try:
        wrapped_run_server(port=port, colab=False, extract_to=data_dir, extract_to2=output_dir)
    except Exception as e:
        print(f"Error in server thread: {e}")

# Start server in a background daemon thread so it doesn't block cell execution
server_thread = threading.Thread(target=run_headless_server, daemon=True)
server_thread.start()

time.sleep(2)
print("Server is now running in the background!")
print(f"If you have setup an SSH tunnel (e.g. ngrok), connect it to port: {port}")

In [ ]:
# @title 4. Tensorboard
import os

output_dir = "/kaggle/working/output"
PORT=18002

# Load inline TensorBoard if supported by browser
%load_ext tensorboard
%tensorboard --logdir={output_dir} --port {PORT}
print(f"TensorBoard launched on port {PORT}.")

In [ ]:
# @title 5. Start Training
training_steps = 5000 # @param {type:"integer"}
batch_size = 8 # @param {type:"integer"}
learning_rate = 0.00005 # @param {type:"number"}
save_every_n_steps = 1000 # @param {type:"integer"}
warmup_steps = 1000 # @param {type:"integer"}
num_workers = 8 # @param {type:"integer"}
resume_from_checkpoint = True # @param {type:"boolean"}

import json
import os
import shutil
import re
import torchaudio
import sys
import glob

# Force patch the actual torchaudio module file for backend compatibility
torchaudio_path = torchaudio.__file__
if "__init__.py" in torchaudio_path:
    with open(torchaudio_path, "r") as f:
        content = f.read()
    if "list_audio_backends" not in content:
        with open(torchaudio_path, "a") as f:
            f.write("\ndef list_audio_backends(): return ['soundfile']\n")

# Ensure paths
working_dir = "/kaggle/working/beatrice-trainer"
data_dir = "/kaggle/working/data"
output_dir = "/kaggle/working/output"
%cd {working_dir}

# Setup config file
config_path = os.path.join(working_dir, "training_config.json")
config_search = glob.glob("**/default_config.json", recursive=True)
if config_search:
    shutil.copy2(config_search[0], config_path)
    with open(config_path, "r") as f:
        data = json.load(f)
    data["n_steps"] = training_steps
    data["batch_size"] = batch_size
    data["learning_rate_g"] = learning_rate
    data["learning_rate_d"] = learning_rate
    data["save_interval"] = save_every_n_steps
    data["evaluation_interval"] = save_every_n_steps
    data["warmup_steps"] = warmup_steps
    data["num_workers"] = num_workers
    with open(config_path, "w") as f:
        json.dump(data, f, indent=4)
else:
    with open(config_path, 'w') as f:
        json.dump({
            "n_steps": training_steps, 
            "batch_size": batch_size, 
            "learning_rate_g": learning_rate,
            "learning_rate_d": learning_rate,
            "save_interval": save_every_n_steps,
            "evaluation_interval": save_every_n_steps,
            "warmup_steps": warmup_steps,
            "num_workers": num_workers
        }, f)

# Check for latest checkpoint to resume training
resume_flag = ""
if resume_from_checkpoint:
    speaker_dirs = [d for d in glob.glob(os.path.join(output_dir, "*")) if os.path.isdir(d)]
    checkpoint_found = False
    for s_dir in speaker_dirs:
        latest_cp = os.path.join(s_dir, "checkpoint_latest.pt.gz")
        if os.path.exists(latest_cp):
            print(f"🔄 Latest checkpoint found at: {latest_cp}. Resuming training...")
            resume_flag = "-r"
            checkpoint_found = True
            break
    if not checkpoint_found:
        print("⚠️ No checkpoint found to resume from. Starting fresh training...")

print(f"Starting training for {training_steps} steps...")
!PYTHONPATH={working_dir} {sys.executable} -m beatrice_trainer -d {data_dir} -o {output_dir} -c {config_path} {resume_flag}

In [ ]:
# @title 6. Download Output
download_type = "VST3 Paraphernalia (Recommended - Fast)" # @param ["VST3 Paraphernalia (Recommended - Fast)", "Latest Checkpoint (.pt.gz)", "Entire Output Folder (Slow)"]

import os
import glob
import shutil
from IPython.display import FileLink, display

output_path = "/kaggle/working/output"

# Helper function to zip and output FileLink
def generate_download_link(folder_to_zip, zip_name):
    zip_file = f"/kaggle/working/{zip_name}.zip"
    if os.path.exists(zip_file):
        os.remove(zip_file)
    print(f"📦 Zipping {folder_to_zip}...")
    shutil.make_archive(f"/kaggle/working/{zip_name}", 'zip', folder_to_zip)
    print(f"🔗 Click the link below to download your files:")
    display(FileLink(zip_file))

if not os.path.exists(output_path) or not os.listdir(output_path):
    print("❌ No outputs found! Please run training first.")
else:
    if "VST3 Paraphernalia" in download_type:
        # Find any paraphernalia folders (either directly in output or inside a subfolder)
        paraphernalia_dirs = sorted(glob.glob(os.path.join(output_path, "**/paraphernalia_*"), recursive=True))
        if paraphernalia_dirs:
            latest_p = paraphernalia_dirs[-1]
            print(f"📂 Found latest paraphernalia folder: {latest_p}")
            generate_download_link(latest_p, f"beatrice_vst3_files")
        else:
            print("❌ No paraphernalia folders found. Did training save checkpoints?")
            
    elif "Latest Checkpoint" in download_type:
        # Find any checkpoint_latest.pt.gz (directly or inside a subfolder)
        latest_cp_files = sorted(glob.glob(os.path.join(output_path, "**/checkpoint_latest.pt.gz"), recursive=True))
        if not latest_cp_files:
            # Fallback to any checkpoint_data_*.pt.gz files
            latest_cp_files = sorted(glob.glob(os.path.join(output_path, "**/checkpoint_data_*.pt.gz"), recursive=True))
            
        if latest_cp_files:
            latest_cp = latest_cp_files[-1]
            print(f"📂 Found latest checkpoint file: {latest_cp}")
            target_cp = f"/kaggle/working/beatrice_checkpoint_latest.pt.gz"
            shutil.copy2(latest_cp, target_cp)
            print(f"🔗 Click the link below to download your checkpoint file:")
            display(FileLink(target_cp))
        else:
            print("❌ No checkpoint files (.pt.gz) found.")
            
    else:
        # Zip and download entire output folder
        generate_download_link(output_path, f"beatrice_all_outputs")

In [ ]:
# @title 7. Delete Dataset and Output Files (Clean up)
# @markdown **WARNING:** This will permanently delete your dataset and training outputs to free up Kaggle workspace disk space.
confirm_deletion = False # @param {type:"boolean"}

import os
import shutil

if not confirm_deletion:
    print("⚠️ Please check 'confirm_deletion' checkbox to delete the files.")
else:
    # Delete local beatrice-trainer
    if os.path.exists("/kaggle/working/beatrice-trainer"):
         print("🗑️ Deleting local repo...")
         shutil.rmtree("/kaggle/working/beatrice-trainer")
         
    # Delete dataset folder
    data_dir = "/kaggle/working/data"
    if os.path.exists(data_dir):
        print(f"🗑️ Deleting dataset: {data_dir}")
        shutil.rmtree(data_dir)
        
    # Delete output folder
    output_dir = "/kaggle/working/output"
    if os.path.exists(output_dir):
        print(f"🗑️ Deleting outputs: {output_dir}")
        shutil.rmtree(output_dir)
        
    # Delete leftover ZIPs
    for file in glob.glob("/kaggle/working/*.zip"):
        os.remove(file)
        
    print("✨ Cleanup complete! Working directory files deleted.")